## Pricing Structure and Seller Performance on Shopee Taiwan

### 1. Setup

In [1]:
import pandas as pd
import numpy as np
import os

### 2. Data Ingestion
Load the last 19 days of daily Shopee product listing snapshots for Taiwan from HuggingFace, including variant pricing, seller metrics, ratings, and shipping data. Each file is one day's snapshot (~1,000 rows). I concatenate them into a single raw DataFrame.

In [2]:
LOCAL_PATH = "data/shopee_products.parquet"

if not os.path.exists(LOCAL_PATH):
    base_url = "https://huggingface.co/datasets/rebrowser/shopee-dataset/resolve/main/product-details/data/"
    dates = pd.date_range("2026-04-25", "2026-05-13")
    all_df = []

    for date in dates:
        file_name = f"{date.strftime('%Y-%m-%d')}.parquet"
        url = base_url + file_name
        try:
            df_temp = pd.read_parquet(url)
            all_df.append(df_temp)
        except Exception as e:
            print(f"Failed: {file_name} | {e}")

    df = pd.concat(all_df, ignore_index=True)
    df.to_parquet(LOCAL_PATH, index=False)
    print(f"{df.shape[0]} rows | {df.shape[1]} columns")
else:
    print("File already exists.")

File already exists.


### 3. Initial Data Audit

In [3]:
df = pd.read_parquet("data/shopee_products.parquet")

In [4]:
# --- 3.1 Shape & Schema ---
print("Shape:")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

print("\nData Types:")
print(df.dtypes.to_string())

Shape:
Rows: 19000 | Columns: 52

Data Types:
_primaryKey                            object
_firstSeenAt              datetime64[ms, UTC]
_lastSeenAt               datetime64[ms, UTC]
itemId                                 object
shopId                                 object
title                                  object
description                            object
status                                 object
condition                               uint8
catId                                  uint32
categories                             object
brand                                  object
brandId                               float64
currency                               object
price                                  object
priceMin                                int64
priceMax                                int64
priceBeforeDiscount                    object
priceMinBeforeDiscount                 object
priceMaxBeforeDiscount                 object
discount                          

In [5]:
# --- 3.2 Missing Values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_report = missing_report[missing_report['count'] > 0].sort_values('pct', ascending=False)

print("Missing Values:")
print(missing_report.to_string())

Missing Values:
                      count     pct
isIndividualSeller    19000  100.00
isMart                19000  100.00
brand                 12362   65.06
brandId               12361   65.06
shopRating             2104   11.07
description            1118    5.88
shopDetailedLocation    667    3.51
shopLocation            651    3.43
shopResponseRate        268    1.41
shippingFeeMin           66    0.35


In [6]:
# --- 3.3 Duplicates Check ---
true_dupes = df.duplicated(subset=['itemId']).sum()
print(f"Duplicate itemId : {true_dupes:,}")
print(f"Unique itemId    : {df['itemId'].nunique():,}")
print(f"Unique shopId    : {df['shopId'].nunique():,}")

Duplicate itemId : 0
Unique itemId    : 19,000
Unique shopId    : 13,342


In [7]:
# --- 3.4 Market & Status Check ---
print(df['currency'].value_counts().to_string())
print("")

print(df['status'].value_counts().to_string())
print("")

print(df['condition'].value_counts().to_string())

currency
TWD    19000

status
normal    15399
new        3601

condition
1    18928
4       66
0        6


In [8]:
# --- 3.5 Price Sanity Check ---
# Raw / 100,000 = actual TWD
df['priceMin'] = df['priceMin'] / 100_000
df['priceMax'] = df['priceMax'] / 100_000

print("Actual TWD Price Distribution:")
print(df[['priceMin', 'priceMax', 'discount']].describe())

print(f"\npriceMin <= 0 (suspicious): {(df['priceMin'] <= 0).sum()}")
print(f"priceMin >= 500,000 (outlier): {(df['priceMin'] > 500_000).sum()}")
print(f"priceMin == priceMax (no variants): {(df['priceMin'] == df['priceMax']).mean():.2%}")

print(f"\nPreview price:")
print(f"Min    : TWD {df['priceMin'].min()}")
print(f"Median : TWD {df['priceMin'].median()}")
print(f"Max    : TWD {df['priceMin'].max()}")

Actual TWD Price Distribution:
            priceMin       priceMax      discount
count   19000.000000   19000.000000  19000.000000
mean     1260.586105    1646.107789     15.323895
std      6310.812844    9388.167952     23.092769
min         1.000000       1.000000      0.000000
25%       129.000000     165.000000      0.000000
50%       319.000000     390.000000      0.000000
75%       790.000000     920.000000     30.000000
max    326000.000000  381481.000000    100.000000

priceMin <= 0 (suspicious): 0
priceMin >= 500,000 (outlier): 0
priceMin == priceMax (no variants): 65.63%

Preview price:
Min    : TWD 1.0
Median : TWD 319.0
Max    : TWD 326000.0


In [9]:
# --- 3.6 Business Snapshot ---
print("Engagement Metrics:")
print(df[['ratingAvg', 'ratingCount', 'commentCount', 'likedCount']].describe())

print("\nShop Metrics:")
print(df[['shopRating', 'shopResponseRate', 'shopFollowerCount', 'shopItemCount']].describe())

print('\nKey Flags:')
for col in ['isOfficialShop', 'isFreeShipping', 'isShopeeVerified', 'isPreOrder']:
    pct = df[col].value_counts(normalize=True).mul(100).round(2)
    print("")
    print(pct.to_string())

Engagement Metrics:
          ratingAvg   ratingCount  commentCount    likedCount
count  19000.000000  19000.000000  19000.000000  19000.000000
mean       1.363147     26.986474     25.424000     13.068895
std        2.212087    388.099402    379.560518    131.387394
min        0.000000      0.000000      0.000000      0.000000
25%        0.000000      0.000000      0.000000      0.000000
50%        0.000000      0.000000      0.000000      0.000000
75%        4.867051      1.000000      1.000000      1.000000
max        5.000000  35639.000000  35308.000000   7164.000000

Shop Metrics:
         shopRating  shopResponseRate  shopFollowerCount  shopItemCount
count  16896.000000      18732.000000       1.900000e+04   19000.000000
mean       4.914722         75.887946       1.581431e+04    3927.108526
std        0.161781         20.525125       9.372038e+04   24025.472365
min        0.000000          1.000000       0.000000e+00       0.000000
25%        4.878672         57.000000       4.1

### 4. Structural Cleaning

In [10]:
# --- 4.1 Drop Unused Columns ---
DROP_COLS = [
    # Premium/locked
    'price', 'priceBeforeDiscount', 'priceMinBeforeDiscount',
    'priceMaxBeforeDiscount', 'models', 'productUrl',
    # Zero-fill
    'isIndividualSeller', 'isMart',
    # Non-analytical
    'description', 'images', 'categories',
    # Redundant / internal
    'brandId', '_primaryKey'
]

df = df.drop(columns=DROP_COLS, errors='ignore')
print(f'Columns after drop: {df.shape[1]}')

Columns after drop: 39


In [11]:
# --- 4.2 Type Coercion ---
DATETIME_COLS = ['_firstSeenAt', '_lastSeenAt', 'createdAt', 'shopCreatedAt']
for col in DATETIME_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)

FLOAT_COLS = [
    'priceMin', 'priceMax', 'shippingFeeMin', 
    'ratingAvg', 'shopRating', 'shopResponseRate', 'discount'
]
for col in FLOAT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')

print(df.dtypes.to_string())

_firstSeenAt            datetime64[ms]
_lastSeenAt             datetime64[ms]
itemId                          object
shopId                          object
title                           object
status                          object
condition                        uint8
catId                           uint32
brand                           object
currency                        object
priceMin                       float64
priceMax                       float64
discount                       float64
ratingAvg                      float64
ratingCount                     uint32
commentCount                    uint32
likedCount                      uint32
shopLocation                    object
isAdult                           bool
isPreOrder                        bool
estimatedDays                    uint8
isFreeShipping                    bool
isServiceByShopee                 bool
createdAt               datetime64[ms]
shopName                        object
shopRating               

In [ ]:
# --- 4.3 Categorical Null Fill ---
df['brand'] = df['brand'].fillna('Unknown')

print(f"brand 'Unknown'      : {(df['brand'] == 'Unknown').sum():,}")
print(f'shopLocation null    : {df["shopLocation"].isnull().sum():,}')
print(f'shopDetailedLocation : {df["shopDetailedLocation"].isnull().sum():,}')

brand 'Unknown'      : 12,362
shopLocation null    : 651
shopDetailedLocation : 667


### 5. Pre-Export Validation

In [13]:
# --- 5.1 Final Shape ---
print(f'Final shape: {df.shape[0]} rows | {df.shape[1]} columns')

# --- 5.2 Critical Column Null Check ---
critical_cols = ['itemId', 'shopId', 'priceMin', 'priceMax', 'discount', 'ratingAvg', 'status']
print('\nNull check:')
print(df[critical_cols].isnull().sum().to_string())

# --- 5.3 Price Range Preview (converted) ---
print('\nPrice range TWD (Actual):')
print(f'Min    : TWD {df["priceMin"].min():>10,.0f}')
print(f'Median : TWD {df["priceMin"].median():>10,.0f}')
print(f'Mean   : TWD {df["priceMin"].mean():>10,.0f}')
print(f'Max    : TWD {df["priceMin"].max():>10,.0f}')

# --- 5.4 Status Distribution ---
print('\nStatus distribution:')
print(df['status'].value_counts().to_string())

Final shape: 19000 rows | 39 columns

Null check:
itemId       0
shopId       0
priceMin     0
priceMax     0
discount     0
ratingAvg    0
status       0

Price range TWD (Actual):
Min    : TWD          1
Median : TWD        319
Mean   : TWD      1,261
Max    : TWD    326,000

Status distribution:
status
normal    15399
new        3601


### 6. Export SQL-Ready CSV

In [14]:
OUTPUT_PATH = 'data/shopee_products_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)